### Importing Logging and SmartConnect

In [1]:
from SmartApi import SmartConnect
from SmartApi.loggerConfig import get_logger 

logger = get_logger(__name__, "INFO")

#### Define client info

In [ ]:
client_info = {
    "api_key": "",
    "client_id": "",
    "password": "",
    "totp_secret": "",
}

#### Generate SmartApi session

In [ ]:
import pyotp

try:
    totp = pyotp.TOTP(client_info["totp_secret"]).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e

api = SmartConnect(api_key=client_info["api_key"])

response  = api.generateSession(client_info["client_id"], client_info["password"], totp)
response

### WebSocket Streaming

##### [SmartAPI WebSocket Streaming 2.0 Documentation](https://smartapi.angelbroking.com/docs/WebSocket2)

#### Packages to Import

In [4]:
import threading
from collections import deque
from SmartApi import SmartWebSocketV2

#### Initialize the smartapi Websocket

In [ ]:
sws = SmartWebSocketV2(
    api_key=client_info["api_key"],
    auth_token = api.getaccessToken,
    client_code = client_info["client_id"],
    feed_token = api.getfeedToken
)

##### This line initializes a thread-safe, fixed-size message cache using a deque (with a maximum of 500 messages) and a Lock to ensure safe access from multiple threads.

In [ ]:
# === Message cache ===
MAX_CACHE_SIZE = 500

cache = deque(maxlen=MAX_CACHE_SIZE)
cache_lock = threading.Lock()

#### Define tokens for symbols and mode (customized)

In [ ]:
correlation_id = "abc123"
mode = 2 # 1 (LTP), 2 (Quote), 3 (Snap Quote)

symbols_data = [
    {
        "name": "NIFTY",
        "token": "99926000"
    },
    {
        "name": "BANKNIFTY",
        "token": "99926009"
    },
]

token_list = [
    {
        "exchangeType": 1,
        "tokens": [symbol["token"] for symbol in symbols_data]
    }
]

#### Sets up a WebSocket client with logging, event handling, and safe message caching.

In [ ]:
def on_data(wsapp, message: dict):
    logger.info(f"Received message: {message}")
    with cache_lock:
        cache.append(message)

def on_open(wsapp):
    logger.info("WebSocket connection opened.")
    sws.subscribe(correlation_id, mode, token_list)
    logger.info(f"Subscribed to tokens: {token_list}")

def on_error(wsapp, error):
    logger.error(f"WebSocket error: {error}")

def on_close(wsapp):
    logger.info("WebSocket connection closed.")


# === Close WebSocket ===
def close_ws():
    logger.info("Closing WebSocket connection...")
    sws.close_connection()

# === Graceful Shutdown on Ctrl+C ===
def handle_exit(sig, frame):
    logger.info("SIGINT (Ctrl+C) detected. Exiting...")
    close_ws()
    exit(0)


# === Connect WebSocket ===
def connect():
    logger.info("Attempting to connect to WebSocket...")
    
    # Assign callbacks
    sws.on_open = on_open
    sws.on_data = on_data
    sws.on_error = on_error
    sws.on_close = on_close

    sws.connect()
    logger.info("WebSocket connection initiated.")


#### Starts the connect function in a background thread as a daemon, allowing the main program to continue running while the WebSocket runs in parallel.

In [8]:
threading.Thread(target=connect, daemon=True).start()

2025-09-06 14:13:52,306 - __main__ - INFO - WebSocket opened.
2025-09-06 14:13:52,308 - __main__ - INFO - Subscribed to: [{'exchangeType': 1, 'tokens': ['99926000', '99926009']}]


### You can use the cached data for processing, analysis, or storage


In [12]:
cache

deque([{'subscription_mode': 2,
        'exchange_type': 1,
        'token': '99926000',
        'sequence_number': 0,
        'exchange_timestamp': 1757068691000,
        'last_traded_price': 2474100,
        'subscription_mode_val': 'QUOTE',
        'last_traded_quantity': 0,
        'average_traded_price': 0,
        'volume_trade_for_the_day': 0,
        'total_buy_quantity': 0.0,
        'total_sell_quantity': 0.0,
        'open_price_of_the_day': 2481885,
        'high_price_of_the_day': 2483235,
        'low_price_of_the_day': 2462160,
        'closed_price': 2473430},
       {'subscription_mode': 2,
        'exchange_type': 1,
        'token': '99926009',
        'sequence_number': 0,
        'exchange_timestamp': 1757068691000,
        'last_traded_price': 5411455,
        'subscription_mode_val': 'QUOTE',
        'last_traded_quantity': 0,
        'average_traded_price': 0,
        'volume_trade_for_the_day': 0,
        'total_buy_quantity': 0.0,
        'total_sell_quantity'

In [10]:
def get_symbol_name(token: str):
    for data in symbols_data:
        if data["token"] == token:
            return data["name"]

In [13]:
for item in cache:
    print(
        f"{get_symbol_name(item['token'])}: "
        f"LTP - {item['last_traded_price']/100}, Closed Price - {item['closed_price']/100}"
    )

NIFTY: LTP - 24741.0, Closed Price - 24734.3
BANKNIFTY: LTP - 54114.55, Closed Price - 54075.45
